# Advanced: Comparison with HALO-(AC)³ aircraft measurements (solar)

Here, we simulate the irradiance at 15 output altitudes along the flight path of the HALO aircraft during the HALO-(AC)³ campaign. The timesteps are filtered for cloud-free conditions and the simulated albedo is compared to aircraft observations over Arctic sea ice.

In [8]:
# Import PyRadtran components
from pyradtran.interface import PyRadtranAccessor  # This should register the accessor automatically
from dataclasses import asdict
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from pathlib import Path
import os
import yaml
import pandas as pd

# Load the configuration from the YAML file
config_path = Path('config/radiosonde.yaml')
import logging
# Configure logging for pyradtran
logging.getLogger('pyradtran').setLevel(logging.CRITICAL)

In [9]:
ds = xr.open_dataset('data/HALO-AC3_HALO_P5_P6_aircraft_broadband_radiation_clear_sky_with_ocean_100s.csv').interpolate_na('time')
# the albedo has over 30% missing values, so we interpolate it for now

ValueError: did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'scipy', 'ee', 'rasterio', 'zarr']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

In [ ]:
# Run a spectral simulation for the dataset
print("Running batch spectral simulation...")
ds_sim = ds.pyradtran.run(
    config_path=config_path,
    return_dataset=True,
    save_to_file=True,
    output_path='data/Simulated_HALO-AC3_HALO_aircraft_broadband_radiation_clear_sky_with_ocean_600s.nc',
    albedo_var='albedo',
)

print("\nSimulation complete!")
ds_sim

## Simulation Results

Spectral radiative transfer calculations completed using DISORT solver with multi-level output altitude grid.

In [ ]:
albedo_sim_z0 = ds_sim.albedo.isel(altitude=0)
albedo_sim_z10 = ds_sim.albedo.isel(altitude=10)
albedo_meas = ds.albedo

fig, (ax, ax_scatter) = plt.subplots(1, 2, gridspec_kw={'width_ratios': [2, 1]}, figsize=(12, 4))
ax.plot(albedo_sim_z10, label='Simulated Albedo at 10m', linestyle='-', marker='x', color='black', alpha=0.7)
ax.plot(albedo_meas, label='Measured Albedo', linestyle='--', marker='o', color='blue', alpha=0.7)
ax.plot(albedo_sim_z0, label='Simulated Albedo at 0m', linestyle='-', marker='s', color='green', alpha=0.7  )
ax.grid(alpha=0.3)
ax.set_xlabel('Time')
ax.set_ylabel('Albedo')
ax.legend(loc='upper right')

ax_scatter.scatter(albedo_meas, albedo_sim_z0, label='sim zout 0m', color='green', alpha=0.5)
ax_scatter.scatter(albedo_meas, albedo_sim_z10, label='sim zout 10m', color='black', alpha=0.5)
ax_scatter.plot([0, 1], [0, 1], linestyle='--', color='k', alpha=0.5)
ax_scatter.set_xlabel('Measured Albedo')
ax_scatter.set_ylabel('Simulated Albedo')
ax_scatter.legend(loc='upper left')
